# AdaptiveDepth Phase 5 targeted correction
This Colab notebook preserves the original step-500 checkpoint, performs a four-step smoke run, then optionally runs one bounded 100-step correction and the unchanged Phase 5 acceptance gate. Use a T4 GPU runtime.

In [ ]:
import base64, json, os, pathlib, shutil, subprocess, sys, zipfile

REPO_URL = 'https://github.com/VedantPandhare/AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs.git'
base = pathlib.Path('/content')
repo = base / 'AdaptiveDepth-Dynamic-Layer-Routing-for-LLMs'

def read_secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

if not repo.exists():
    clone_env = os.environ.copy()
    github_token = read_secret('GITHUB_TOKEN')
    if github_token:
        credential = base64.b64encode(f'x-access-token:{github_token}'.encode()).decode()
        clone_env.update({'GIT_CONFIG_COUNT': '1', 'GIT_CONFIG_KEY_0': 'http.extraHeader', 'GIT_CONFIG_VALUE_0': f'Authorization: Basic {credential}'})
    clone = subprocess.run(['git', 'clone', REPO_URL, str(repo)], env=clone_env, capture_output=True, text=True)
    if clone.returncode:
        detail = clone.stderr.strip().splitlines()[-1] if clone.stderr.strip() else 'unknown Git error'
        raise RuntimeError(f'Git clone failed: {detail}\nFor a private repository, add a read-only GITHUB_TOKEN to Colab Secrets.')
os.chdir(repo)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.[data]'], check=True)
subprocess.run(['nvidia-smi'], check=True)
import torch
assert torch.cuda.is_available(), 'Select a GPU runtime before running the correction'
print('Repository:', repo)
print('GPU:', torch.cuda.get_device_name(0))

## Mount Drive and stage the base checkpoint
Upload the complete `checkpoint-step-000500` folder to My Drive. Change `DRIVE_CHECKPOINT` if it is stored in a subfolder. The original Drive folder is read-only for this experiment.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_CHECKPOINT = pathlib.Path('/content/drive/MyDrive/checkpoint-step-000500')
LOCAL_CHECKPOINT = pathlib.Path('/content/checkpoint-step-000500')
required = ['model.safetensors', 'config.json', 'tokenizer.json', 'tokenizer_config.json']
missing = [name for name in required if not (DRIVE_CHECKPOINT / name).is_file()]
if missing:
    raise FileNotFoundError(f'Incomplete checkpoint at {DRIVE_CHECKPOINT}; missing: {missing}')
source_size = (DRIVE_CHECKPOINT / 'model.safetensors').stat().st_size
if source_size < 1.5 * 1024**3:
    raise RuntimeError(f'model.safetensors is unexpectedly small: {source_size / 1024**3:.3f} GB')
if LOCAL_CHECKPOINT.exists():
    shutil.rmtree(LOCAL_CHECKPOINT)
shutil.copytree(DRIVE_CHECKPOINT, LOCAL_CHECKPOINT)
if (LOCAL_CHECKPOINT / 'model.safetensors').stat().st_size != source_size:
    raise RuntimeError('Local checkpoint copy is incomplete')
print('Base checkpoint staged without modification:', LOCAL_CHECKPOINT)

## Choose smoke or full correction
Keep `RUN_FULL = False` first. After the smoke succeeds, set it to `True` and rerun this cell and the final two cells.

In [ ]:
RUN_FULL = False
if RUN_FULL:
    dataset = pathlib.Path('data/phase5_correction_train.jsonl')
    if not dataset.exists():
        subprocess.run([
            'adaptivedepth', 'prepare-phase5-correction-data',
            '--output', str(dataset), '--dolly-examples', '300',
            '--balanced-examples', '200', '--skip-dolly-examples', '2000',
        ], check=True)
    config = 'configs/phase5_correction_t4.yaml'
    workspace = pathlib.Path('experiments/phase5_correction')
else:
    dataset = pathlib.Path('data/phase5_correction_smoke.jsonl')
    config = 'configs/phase5_correction_smoke.yaml'
    workspace = pathlib.Path('experiments/phase5_correction_smoke')
print('Mode:', 'FULL' if RUN_FULL else 'SMOKE')
print('Dataset:', dataset, 'examples:', sum(1 for _ in dataset.open(encoding='utf-8')))
print('Workspace:', workspace)

In [ ]:
command = [
    'adaptivedepth', 'run-phase5-correction',
    '--checkpoint', str(LOCAL_CHECKPOINT),
    '--config', config,
    '--routes-file', 'configs/phase5_correction_route.yaml',
    '--dataset', str(dataset),
    '--output', str(workspace),
    '--resume',
]
subprocess.run(command, check=True)

In [ ]:
summary_path = workspace / 'training_summary.json'
summary = json.loads(summary_path.read_text())
checkpoint = pathlib.Path(summary['final_checkpoint'])
archive = pathlib.Path('phase5_corrected_student.zip' if RUN_FULL else 'phase5_correction_smoke_student.zip')
with zipfile.ZipFile(archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
    for path in checkpoint.rglob('*'):
        if path.is_file() and path.name != 'training_state.pt':
            bundle.write(path, pathlib.Path(checkpoint.name) / path.relative_to(checkpoint))
    bundle.write(summary_path, 'training_summary.json')
drive_results = pathlib.Path('/content/drive/MyDrive/AdaptiveDepth-results')
drive_results.mkdir(parents=True, exist_ok=True)
shutil.copy2(archive, drive_results / archive.name)
shutil.copy2(summary_path, drive_results / ('phase5_correction_training_summary.json' if RUN_FULL else 'phase5_correction_smoke_summary.json'))
print('Correction checkpoint:', checkpoint)
print('Saved model archive:', drive_results / archive.name)

if not RUN_FULL:
    print('Smoke correction complete. Set RUN_FULL = True and rerun the final three code cells.')
else:
    evaluation_workspace = pathlib.Path('experiments/phase5_correction_evaluation')
    subprocess.run([
        'adaptivedepth', 'run-phase5-evaluation',
        '--checkpoint', str(checkpoint),
        '--config', 'configs/cloud_t4.yaml',
        '--routes-file', 'configs/phase5_routes.yaml',
        '--dataset', 'data/phase4_eval.jsonl',
        '--reference-file', 'data/phase4_reference_metrics.json',
        '--evaluation-config', 'configs/phase5_evaluation.yaml',
        '--workspace', str(evaluation_workspace), '--resume',
    ], check=True)
    print((evaluation_workspace / 'gate.md').read_text())
    evaluation_archive = pathlib.Path('phase5_correction_evaluation_artifacts.zip')
    with zipfile.ZipFile(evaluation_archive, 'w', compression=zipfile.ZIP_DEFLATED) as bundle:
        for path in evaluation_workspace.rglob('*'):
            if path.is_file():
                bundle.write(path, evaluation_workspace.name + '/' + str(path.relative_to(evaluation_workspace)))
    shutil.copy2(evaluation_archive, drive_results / evaluation_archive.name)
    shutil.copy2(evaluation_workspace / 'gate.md', drive_results / 'phase5_correction_gate.md')
    shutil.copy2(evaluation_workspace / 'gate.json', drive_results / 'phase5_correction_gate.json')
    print('Saved correction evaluation:', drive_results / evaluation_archive.name)